In [102]:
#%pip install itables

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.0 MB/s eta 0:00:00


In [97]:
# Install ITables once if it is not already available.
# Uncomment and run this in a separate notebook cell if required:
#%pip install itables

from itables import show

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

# Retrieve the active Spark session or create one.
spark = SparkSession.builder.getOrCreate()

# Define the raw source file name and location.
volume_raw_whole_fleet_vehicle_registrations = (
    "/Volumes/transport_planning/default/vehicle_registrations/"
    "whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv"
)

# Define the bronze table name and location.
table_bronze_whole_fleet_vehicle_registrations = (
    "transport_planning.default."
    "bronze_whole_fleet_vehicle_registration_by_postcode_2026_q2"
)

# Read the CSV and infer the schema.
bronze_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(volume_raw_whole_fleet_vehicle_registrations)
)


# Write the raw records to the managed Bronze Delta table.
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(table_bronze_whole_fleet_vehicle_registrations)
)


# Read the persisted table so verification checks the saved result
# rather than only the source DataFrame.
bronze_result_df = spark.table(
    table_bronze_whole_fleet_vehicle_registrations
)


# Verify the persisted schema and record count.
bronze_result_df.printSchema()

bronze_row_count = bronze_result_df.count()
print(f"Bronze rows written: {bronze_row_count:,}")


# Render a small interactive preview in the notebook UI.
show(bronze_result_df.limit(20).toPandas())

root
 |-- CD_MAKE_VEH1: string (nullable = true)
 |-- CD_CLASS_VEH: integer (nullable = true)
 |-- NB_YEAR_MFC_VEH: integer (nullable = true)
 |-- POSTCODE: integer (nullable = true)
 |-- CD_CL_FUEL_ENG: string (nullable = true)
 |-- TOTAL1: integer (nullable = true)



Bronze rows written: 1,156,741


<!--| quarto-html-table-processing: none -->
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

 
 
 
 

 
 
 
 

 
 
 
 

 
 
 
 
 
 
 
 
 Loading ITables v2.9.1 from the internet...
 (need help ?)
 
 
 
 
 
 🔒 ⓘ CD_MAKE_VEH1 
 CD_CLASS_VEH 
 NB_YEAR_MFC_VEH 
 POSTCODE 
 CD_CL_FUEL_ENG 
 TOTAL1 
 
 A AUST 2 1998 3431 D 1 
 A BARF 2 1967 3302 D 1 
 A BARF 2 1975 3352 D 1 
 A BARF 2 1981 3414 D 1 
 A BARF 2 1965 3517 P 1 
 A BARF 2 1986 3550 D 1 
 A BARF 2 1987 3550 D 1 
 A BARF 2 1969 3844 D 1 
 A BARF 2 1974 3873 D 1 
 A BARF 2 1981 3875 D 1 
 (10 more rows not shown)

## Silver Transformation

The Silver transformation converts the Bronze whole-fleet vehicle registration snapshot into a standardised, quality-controlled and analysis-ready Delta table. It cleans coded values, consolidates duplicate registration segments, adds snapshot and data-quality metadata, and verifies that the total number of registered vehicles is preserved.

The transformation includes:

- Renaming source columns using descriptive snake-case names.
- Removing the byte-order mark and trimming whitespace from coded values.
- Converting blank fuel codes to `UNKNOWN`.
- Treating postcodes and classification codes as string identifiers.
- Converting manufacture year and registration counts to appropriate numeric types.
- Converting manufacture year `0` to null and flagging year `1900` for review.
- Removing records with missing required fields, invalid years or non-positive registration counts.
- Consolidating duplicate make, class, year, postcode and fuel combinations.
- Summing registration counts while preserving the source total.
- Adding snapshot date, year, quarter and reporting-period fields.
- Calculating vehicle age and analysis-friendly age bands.
- Classifying postcodes as Victorian, interstate, administrative, unknown or invalid.
- Adding a deterministic registration-segment key.
- Adding source-file and Silver-processing timestamps for traceability.
- Writing the validated result to the managed `silver_whole_fleet_vehicle_registrations_by_postcode` Delta table.
- Reconciling Bronze and Silver registration totals after processing.

Vehicle, fuel and postcode descriptions are not inferred during this transformation. They should be added through authoritative reference tables in a later enrichment step.

In [104]:
from itables import show

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# Retrieve the active Spark session or create one.
spark = SparkSession.builder.getOrCreate()


# Define the fully qualified Bronze source table.
table_bronze_whole_fleet_vehicle_registrations = (
    "transport_planning.default."
    "bronze_whole_fleet_vehicle_registration_by_postcode_2026_q2"
)

# Define the fully qualified Silver destination table.
table_silver_whole_fleet_vehicle_registrations = (
    "transport_planning.default."
    "silver_whole_fleet_vehicle_registrations_by_postcode"
)

# Source metadata for this registration snapshot.
source_file_name = (
    "whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv"
)

snapshot_date = "2026-06-30"
snapshot_year = 2026
snapshot_quarter = 2
snapshot_period = "2026-Q2"


# Read the persisted Bronze Delta table.
bronze_df = spark.table(
    table_bronze_whole_fleet_vehicle_registrations
)

In [105]:
# Remove a possible UTF-8 byte-order mark and whitespace from column names.
# This handles the BOM observed on the first source column.
for original_column_name in bronze_df.columns:
    clean_column_name = (
        original_column_name
        .replace("\ufeff", "")
        .strip()
    )

    if original_column_name != clean_column_name:
        bronze_df = bronze_df.withColumnRenamed(
            original_column_name,
            clean_column_name,
        )


def blank_to_null(column_name):
    """
    Convert a column to trimmed text and replace blank values with null.
    """
    trimmed_value = F.trim(
        F.col(column_name).cast("string")
    )

    return F.when(
        F.length(trimmed_value) == 0,
        F.lit(None),
    ).otherwise(trimmed_value)


# Prepare postcode as a string before normalisation.
postcode_text = blank_to_null("POSTCODE")

# Restore leading zeroes for one-to-four digit numeric postcodes.
# Values longer than four characters are retained so they are not truncated.
normalised_postcode = F.when(
    postcode_text.rlike(r"^[0-9]{1,4}$"),
    F.lpad(postcode_text, 4, "0"),
).otherwise(postcode_text)

In [106]:
# Rename fields, remove whitespace and enforce Silver data types.
standardised_df = bronze_df.select(
    F.upper(
        blank_to_null("CD_MAKE_VEH1")
    ).alias("vehicle_make_code"),

    F.upper(
        blank_to_null("CD_CLASS_VEH")
    ).alias("vehicle_class_code"),

    F.when(
        F.col("NB_YEAR_MFC_VEH").cast("integer") == 0,
        F.lit(None).cast("integer"),
    ).otherwise(
        F.col("NB_YEAR_MFC_VEH").cast("integer")
    ).alias("manufacture_year"),

    normalised_postcode.alias("postcode"),

    F.when(
        blank_to_null("CD_CL_FUEL_ENG").isNull(),
        F.lit("UNKNOWN"),
    ).otherwise(
        F.upper(blank_to_null("CD_CL_FUEL_ENG"))
    ).alias("fuel_type_code"),

    F.col("TOTAL1")
    .cast("long")
    .alias("registered_vehicle_count"),
)

In [107]:
# Retain records that satisfy the core Silver data-quality requirements.
#
# Unknown manufacture years are retained as null.
# Year 1900 is retained but flagged later for review.
# Non-Victorian postcodes are retained and classified rather than discarded.
valid_df = standardised_df.filter(
    F.col("vehicle_make_code").isNotNull()
    & F.col("vehicle_class_code").isNotNull()
    & F.col("postcode").isNotNull()
    & F.col("registered_vehicle_count").isNotNull()
    & (F.col("registered_vehicle_count") > 0)
    & (
        F.col("manufacture_year").isNull()
        | F.col("manufacture_year").between(
            1900,
            snapshot_year,
        )
    )
)


# Consolidate records that resolve to the same normalised business grain.
#
# The expected grain is one row per:
# snapshot + make + class + manufacture year + postcode + fuel type.
aggregated_df = (
    valid_df
    .groupBy(
        "vehicle_make_code",
        "vehicle_class_code",
        "manufacture_year",
        "postcode",
        "fuel_type_code",
    )
    .agg(
        F.sum("registered_vehicle_count")
        .cast("long")
        .alias("registered_vehicle_count"),

        # Record how many Bronze rows contributed to each Silver record.
        F.count(F.lit(1))
        .cast("long")
        .alias("source_row_count"),
    )
)


# Add reporting-period metadata and basic data-quality classifications.
silver_df = (
    aggregated_df

    # Snapshot metadata
    .withColumn(
        "snapshot_date",
        F.to_date(F.lit(snapshot_date)),
    )
    .withColumn(
        "snapshot_year",
        F.lit(snapshot_year).cast("integer"),
    )
    .withColumn(
        "snapshot_quarter",
        F.lit(snapshot_quarter).cast("integer"),
    )
    .withColumn(
        "snapshot_period",
        F.lit(snapshot_period),
    )

    # Manufacture-year quality classification
    .withColumn(
        "manufacture_year_quality_status",
        F.when(
            F.col("manufacture_year").isNull(),
            F.lit("unknown"),
        )
        .when(
            F.col("manufacture_year") == 1900,
            F.lit("review_required"),
        )
        .otherwise(F.lit("valid")),
    )

    # Calculate vehicle age only for manufacture years considered valid.
    .withColumn(
        "vehicle_age_years",
        F.when(
            F.col("manufacture_year_quality_status") == "valid",
            F.lit(snapshot_year) - F.col("manufacture_year"),
        ).otherwise(
            F.lit(None).cast("integer")
        ),
    )

    # Create analysis-friendly vehicle age bands.
    .withColumn(
        "vehicle_age_band",
        F.when(
            F.col("vehicle_age_years").isNull(),
            F.lit("Unknown"),
        )
        .when(
            F.col("vehicle_age_years").between(0, 2),
            F.lit("0-2 years"),
        )
        .when(
            F.col("vehicle_age_years").between(3, 5),
            F.lit("3-5 years"),
        )
        .when(
            F.col("vehicle_age_years").between(6, 10),
            F.lit("6-10 years"),
        )
        .when(
            F.col("vehicle_age_years").between(11, 15),
            F.lit("11-15 years"),
        )
        .when(
            F.col("vehicle_age_years").between(16, 20),
            F.lit("16-20 years"),
        )
        .otherwise(F.lit("21+ years")),
    )

    # Classify postcodes without discarding interstate or special values.
    .withColumn(
        "postcode_status",
        F.when(
            F.col("postcode") == "0000",
            F.lit("unknown"),
        )
        .when(
            F.col("postcode") == "9000",
            F.lit("administrative"),
        )
        .when(
            F.col("postcode").rlike(r"^3[0-9]{3}$"),
            F.lit("victorian_format"),
        )
        .when(
            F.col("postcode").rlike(r"^[0-9]{4}$"),
            F.lit("interstate_or_other"),
        )
        .otherwise(F.lit("invalid_format")),
    )

    .withColumn(
        "is_victorian_postcode",
        F.col("postcode").rlike(r"^3[0-9]{3}$"),
    )

    # Add source and processing lineage.
    .withColumn(
        "source_file_name",
        F.lit(source_file_name),
    )
    .withColumn(
        "silver_processed_at",
        F.current_timestamp(),
    )
)

In [108]:
# Create a deterministic identifier for the Silver business grain.
silver_df = silver_df.withColumn(
    "registration_segment_key",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("snapshot_period"),
            F.coalesce(
                F.col("vehicle_make_code"),
                F.lit("<NULL>"),
            ),
            F.coalesce(
                F.col("vehicle_class_code"),
                F.lit("<NULL>"),
            ),
            F.coalesce(
                F.col("manufacture_year").cast("string"),
                F.lit("<NULL>"),
            ),
            F.coalesce(
                F.col("postcode"),
                F.lit("<NULL>"),
            ),
            F.coalesce(
                F.col("fuel_type_code"),
                F.lit("<NULL>"),
            ),
        ),
        256,
    ),
)


# Select and order the approved Silver fields.
silver_df = silver_df.select(
    "registration_segment_key",
    "snapshot_date",
    "snapshot_year",
    "snapshot_quarter",
    "snapshot_period",
    "vehicle_make_code",
    "vehicle_class_code",
    "manufacture_year",
    "manufacture_year_quality_status",
    "vehicle_age_years",
    "vehicle_age_band",
    "postcode",
    "postcode_status",
    "is_victorian_postcode",
    "fuel_type_code",
    "registered_vehicle_count",
    "source_row_count",
    "source_file_name",
    "silver_processed_at",
)


# Calculate pre-write reconciliation metrics.
bronze_row_count = bronze_df.count()
valid_source_row_count = valid_df.count()

bronze_registration_total = (
    standardised_df
    .agg(
        F.sum("registered_vehicle_count")
        .alias("registration_total")
    )
    .first()["registration_total"]
)


# Write the curated dataset to the managed Silver Delta table.
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(table_silver_whole_fleet_vehicle_registrations)
)


In [109]:
# Read the persisted Silver table for verification.
silver_result_df = spark.table(
    table_silver_whole_fleet_vehicle_registrations
)

silver_row_count = silver_result_df.count()

silver_registration_total = (
    silver_result_df
    .agg(
        F.sum("registered_vehicle_count")
        .alias("registration_total")
    )
    .first()["registration_total"]
)


# Print verification and reconciliation results.
print(f"Bronze rows:              {bronze_row_count:,}")
print(f"Valid Bronze rows:        {valid_source_row_count:,}")
print(f"Silver rows:              {silver_row_count:,}")
print(
    f"Rows consolidated:        "
    f"{valid_source_row_count - silver_row_count:,}"
)
print(
    f"Bronze registration total: "
    f"{bronze_registration_total:,}"
)
print(
    f"Silver registration total: "
    f"{silver_registration_total:,}"
)
print(
    "Registration totals match: "
    f"{bronze_registration_total == silver_registration_total}"
)


# Display the resulting Silver schema.
silver_result_df.printSchema()


# Render a small interactive preview in the notebook UI.
show(
    silver_result_df
    .limit(20)
    .toPandas()
)

Bronze rows:              1,156,741
Valid Bronze rows:        1,156,741
Silver rows:              924,339
Rows consolidated:        232,402
Bronze registration total: 6,003,177
Silver registration total: 6,003,177
Registration totals match: True
root
 |-- registration_segment_key: string (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- snapshot_year: integer (nullable = true)
 |-- snapshot_quarter: integer (nullable = true)
 |-- snapshot_period: string (nullable = true)
 |-- vehicle_make_code: string (nullable = true)
 |-- vehicle_class_code: string (nullable = true)
 |-- manufacture_year: integer (nullable = true)
 |-- manufacture_year_quality_status: string (nullable = true)
 |-- vehicle_age_years: integer (nullable = true)
 |-- vehicle_age_band: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- postcode_status: string (nullable = true)
 |-- is_victorian_postcode: boolean (nullable = true)
 |-- fuel_type_code: string (nullable = true)
 |-- registe

<!--| quarto-html-table-processing: none -->
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

 
 
 
 

 
 
 
 

 
 
 
 

 
 
 
 
 
 
 
 
 Loading ITables v2.9.1 from the internet...
 (need help ?)
 
 
 
 
 
 🔒 ⓘ registration_segment_key 
 snapshot_date 
 snapshot_year 
 snapshot_quarter 
 snapshot_period 
 vehicle_make_code 
 vehicle_class_code 
 manufacture_year 
 manufacture_year_quality_status 
 vehicle_age_years 
 vehicle_age_band 
 postcode 
 postcode_status 
 is_victorian_postcode 
 fuel_type_code 
 registered_vehicle_count 
 source_row_count 
 source_file_name 
 silver_processed_at 
 
 ef451203be89a39f45f19599c912290c22c11ff1cc4f6ce91bb521e683944766 2026-06-30 2026 2 2026-Q2 A BARF 2 1967 valid 59.0 21+ years 3302 victorian_format True D 1 1 whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv 2026-08-09 07:18:43.770703 
 c36e66a5de70557d35103a304a729bf76bfd0b5f0cac819df2b5f5a8a0d2c6db 2026-06-30 2026 2 2026-Q2 ABARTH 2 2019 valid 7.0 6-10 years 3124 victorian_format True P 1 1 whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv 2026-08-09 07:18:43.770703 
 9d0561a1fc9f4c661728d655c9bc1d9ce0f009647e96ea49f88079f07dc41d97 2026-06-30 2026 2 2026-Q2 AFRON 2 2010 valid 16.0 16-20 years 3350 victorian_format True P 1 1 whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv 2026-08-09 07:18:43.770703 
 7cd5537eccd9e60b2607a0a007f5a9c08ba481e322cd67cf044a0e069b48946b 2026-06-30 2026 2 2026-Q2 AGRIFL 2 2021 valid 5.0 3-5 years 3377 victorian_format True D 1 1 whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv 2026-08-09 07:18:43.770703 
 014d2d6eb33a2a953393288f593ad380f7aaf0b62c9be02d074085761e96c657 2026-06-30 2026 2 2026-Q2 AJP 3 1900 review_required NaN Unknown 3175 victorian_format True P 1 1 whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv 2026-08-09 07:18:43.770703 
 cadcb82c99fa8fd03af2b57f10f9c12f36294569219602b0cdbde08edcc2dade 2026-06-30 2026 2 2026-Q2 ALFA R 2 2024 valid 2.0 0-2 years 3025 victorian_format True M 1 1 whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv 2026-08-09 07:18:43.770703 
 cf8978e7766d66fe46f36dc77fbb87644a6918cb4a92175e9d548d420eb334e7 2026-06-30 2026 2 2026-Q2 ALFA R 2 2007 valid 19.0 16-20 years 3030 victorian_format True D 1 1 whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv 2026-08-09 07:18:43.770703 
 693d3a6092df1999375cc597ffd831f72fb55b8ea6363c3c96b4e4fe4e663c80 2026-06-30 2026 2 2026-Q2 ALFA R 2 2018 valid 8.0 6-10 years 3030 victorian_format True P 7 1 whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv 2026-08-09 07:18:43.770703 
 0664d7ee3c1d2370b3c4b59e438bc0805ebb1fb9f75a1780999b43710400cca2 2026-06-30 2026 2 2026-Q2 ALFA R 2 2023 valid 3.0 3-5 years 3030 victorian_format True M 2 1 whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv 2026-08-09 07:18:43.770703 
 693eeb36dcac68eb068a92c3f881f0931370878dbb30051d93494b7f1062f3e5 2026-06-30 2026 2 2026-Q2 ALFA R 2 2022 valid 4.0 3-5 years 3043 victorian_format True P 1 1 whole_fleet_vehicle_registration_snapshot_by_postcode_q2_2026.csv 2026-08-09 07:18:43.770703 
 (10 more rows not shown)